In [1]:
#@title 🚀 Cell 1: Hardware Environment & Gemini AI Agent Configuration
# Check GPU allocation (Nvidia T4 15-16GB VRAM recommended for Google Colab Free Tier)
!nvidia-smi

import os, sys, psutil

# -------------------------------------------------------------------------
# Google Gemini Vision & AI Director Agent API Key
# -------------------------------------------------------------------------
try:
    from google.colab import userdata
    g_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
    if g_key:
        os.environ['GEMINI_API_KEY'] = g_key
        os.environ['GOOGLE_API_KEY'] = g_key
        print('✅ Google Gemini API Key detected from Colab Secrets.')
    else:
        print('ℹ️ No GEMINI_API_KEY secret found in Colab Secrets. AI Director Agent will run in local procedural mode.')
except Exception:
    pass

try:
    import torch
    print('=' * 60)
    print('CineFlow-AI: System Diagnostic')
    print('=' * 60)
    print(f'Python Version: {sys.version.split()[0]}')
    print(f'PyTorch Version: {torch.__version__}')
    print(f'CUDA Available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU Device: {torch.cuda.get_device_name(0)}')
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f'Total VRAM: {vram_gb:.2f} GB')
        cc_major, cc_minor = torch.cuda.get_device_capability(0)
        print(f'Compute Capability: {cc_major}.{cc_minor} (T4 CC 7.5 Turing Architecture)')
    else:
        print('⚠️ CUDA is not active. Please navigate to Runtime -> Change runtime type -> T4 GPU.')
except ImportError:
    print('PyTorch not yet imported; will be verified after dependency installation.')

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'System Host RAM: {ram_gb:.2f} GB (Ceiling: ~12.7 GB on Colab Free Tier)')
print('=' * 60)


Tue Sep  1 17:56:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#@title 📦 Cell 2: Git Clone & Studio Directory Setup
#@markdown ✨ **Private Repository Support**:
#@markdown If your repository is **Private**, paste your GitHub Personal Access Token (PAT) below, OR set `GITHUB_TOKEN` in Colab Secrets (🔑 on the left sidebar).
#@markdown *(Tip: You can also make your repo **Public** in GitHub Settings → Danger Zone → Change visibility → Public for 1-click clone without any token!)*
GITHUB_PERSONAL_ACCESS_TOKEN = ""  #@param {type:"string"}

import os, sys, getpass, shutil
from pathlib import Path

REPO_URL = 'https://github.com/sohom9143/CineFlowStudio.git'
REPO_NAME = 'CineFlowStudio'
WORKSPACE_DIR = '/content/' + REPO_NAME

# 1. Resolve GitHub Token (Form input > Colab Secrets > None)
gh_token = GITHUB_PERSONAL_ACCESS_TOKEN.strip() if 'GITHUB_PERSONAL_ACCESS_TOKEN' in globals() and GITHUB_PERSONAL_ACCESS_TOKEN else ""
if not gh_token:
    try:
        from google.colab import userdata
        gh_token = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN') or userdata.get('PAT') or ""
    except Exception:
        gh_token = ""

def build_clone_url(token: str) -> str:
    if token:
        return f"https://{token}@github.com/sohom9143/CineFlowStudio.git"
    return REPO_URL

# 2. Clone repository if not already in workspace
if not os.path.exists('modules') and not os.path.exists('app.py'):
    if not os.path.exists(WORKSPACE_DIR):
        print(f"Connecting to CineFlow-AI repository ({REPO_URL})...")
        clone_target_url = build_clone_url(gh_token)
        res = os.system(f"git clone {clone_target_url} {WORKSPACE_DIR}")

        # If clone failed (likely private repo without token), prompt interactively
        if res != 0 or not os.path.exists(WORKSPACE_DIR):
            print("\n⚠️ Anonymous git clone failed. Repository appears to be Private.")
            interactive_token = getpass.getpass("Please enter your GitHub Personal Access Token (PAT): ").strip()
            if interactive_token:
                clone_target_url = build_clone_url(interactive_token)
                res = os.system(f"git clone {clone_target_url} {WORKSPACE_DIR}")

        if res != 0 or not os.path.exists(WORKSPACE_DIR):
            raise RuntimeError(
                "\n" + "=" * 72 + "\n"
                "❌ CLONE FAILED: Repository is Private on GitHub.\n"
                "Please choose one of the following simple solutions:\n\n"
                "  ▶ Solution A (Fastest & Easiest): Make repository Public on GitHub:\n"
                "    1. Visit: https://github.com/sohom9143/CineFlowStudio/settings\n"
                "    2. Scroll to bottom (Danger Zone) → Click 'Change visibility' → Select 'Public'\n\n"
                "  ▶ Solution B: Add a GitHub Personal Access Token (PAT):\n"
                "    1. Create a token at https://github.com/settings/tokens (classic, 'repo' scope)\n"
                "    2. Paste into the 'GITHUB_PERSONAL_ACCESS_TOKEN' box above or add to Colab Secrets (🔑 icon)\n"
                + "=" * 72
            )
        else:
            print("✅ Repository cloned successfully!")
    else:
        print("Repository already present. Pulling latest updates...")
        os.system(f"git -C {WORKSPACE_DIR} pull")

# 3. Permanently switch working directory to CineFlowStudio root
if os.path.exists(WORKSPACE_DIR):
    try:
        get_ipython().run_line_magic('cd', WORKSPACE_DIR)
    except Exception:
        os.chdir(WORKSPACE_DIR)
else:
    print("Already in CineFlow root directory. Pulling latest updates...")
    os.system("git pull")

print(f"Active Studio Directory: {os.getcwd()}")

# 4. Connect Google Drive for 100% Free Persistent JSON Tree & Character Profiles
try:
    from google.colab import drive
    drive_mount_path = '/content/drive'
    if not os.path.exists(drive_mount_path):
        print("💾 Mounting Google Drive for persistent JSON tree & character storage...")
        drive.mount(drive_mount_path, force_remount=False)
    
    drive_studio_dir = '/content/drive/MyDrive/CineFlowStudio'
    os.makedirs(f"{drive_studio_dir}/character_profiles", exist_ok=True)
    os.makedirs(f"{drive_studio_dir}/scene_trees", exist_ok=True)
    print(f"✅ Google Drive storage linked at: {drive_studio_dir}")
    print("   (All JSON trees and character face banks will be permanently saved to your Google Drive!)")
except Exception as drive_err:
    print(f"ℹ️ Google Drive mount skipped ({drive_err}). Using local workspace storage.")

# 5. Initialize all operational pipeline directories
required_dirs = [
    'models',
    'outputs',
    'outputs/masters',
    'outputs/temp',
    'outputs/temp_lipsync',
    'character_profiles',
    'scene_trees',
    'configs',
]
for folder in required_dirs:
    os.makedirs(folder, exist_ok=True)

print("✅ Directory structure initialized successfully.")


In [3]:
#@title 📥 Cell 3: Install Production Dependencies & FFmpeg
import os
if os.path.exists('/content/CineFlowStudio'):
    try:
        get_ipython().run_line_magic('cd', '/content/CineFlowStudio')
    except Exception:
        os.chdir('/content/CineFlowStudio')

# Install FFmpeg system binary for broadcast-grade H.264 / AAC MP4 multiplexing
!apt-get -y update && apt-get install -y ffmpeg

# Upgrade pip and install all pinned dependencies
!pip install --upgrade pip
req_file = 'requirements.txt' if os.path.exists('requirements.txt') else '/content/CineFlowStudio/requirements.txt'
!pip install -r {req_file}
!pip install google-generativeai google-genai

print("✅ All CineFlow-AI requirements installed and verified.")


In [4]:
#@title 🧠 Cell 4: Download Model Weights & Character Face Bank
import os, urllib.request
from tqdm import tqdm

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url: str, output_path: str):
    if os.path.exists(output_path) and os.path.getsize(output_path) > 1024:
        print(f"File already exists: {output_path} ({os.path.getsize(output_path) / (1024*1024):.1f} MB)")
        return
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    print(f"Downloading {os.path.basename(output_path)} from {url}...")
    try:
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(output_path)) as t:
            urllib.request.urlretrieve(url, filename=output_path, reporthook=t.update_to)
    except Exception as e:
        print(f"Note: Download of {os.path.basename(output_path)} failed: {e}. Pipeline will use high-order procedural fallback.")

# Checkpoint download registry
MODEL_REGISTRY = {
    "models/RealESRGAN_x4plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
}

for dest_path, url in MODEL_REGISTRY.items():
    download_file(url, dest_path)

print("✅ Model checkpoints downloaded and Face Bank verified.")


RealESRGAN_x4plus.pth: 67.0MB [00:00, 128MB/s]                            

✅ Model checkpoints downloaded and Face Bank verified.


In [5]:
#@title 🧪 Cell 5: Automated Test Suite Verification (pytest)
import os
if os.path.exists('/content/CineFlowStudio'):
    try:
        get_ipython().run_line_magic('cd', '/content/CineFlowStudio')
    except Exception:
        os.chdir('/content/CineFlowStudio')

# Executes complete diagnostic test suite verifying VRAMManager, CharacterStudio, CineVideoEngine, LipSyncEngine, PostProductionEngine
!pytest tests/test_pipeline_e2e.py tests/test_character_engine.py tests/test_universal_agent.py -v --tb=short


In [6]:
#@title 🎬 Cell 6: Launch CineFlow-AI Studio WebUI (Public Share Link)
import os
if os.path.exists('/content/CineFlowStudio'):
    try:
        get_ipython().run_line_magic('cd', '/content/CineFlowStudio')
    except Exception:
        os.chdir('/content/CineFlowStudio')

# Starts the Beginner-Friendly Gradio Studio with public URL (share=True) for Google Colab remote browser access
!python app.py --share --port 7860 --config configs/colab_t4_config.yaml
